# Step/Ramp Clamp-Target Analysis

This notebook keeps the original data-loading cell intact, converts the recorded LabJack clampTarget voltage back to dF/F target units, extracts step or ramp events by command amplitude, and plots aligned DA traces.

### Import libraries and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyNeuroDAP as ndap
import os
import re
from scipy.signal import butter, filtfilt
from scipy.optimize import least_squares
from scipy import ndimage
import mat73

In [ ]:
# Load data (adjust path as needed)
# root_dir = '/Users/shunli/Projects/BrainClamp/data/'
root_dir = '/Volumes/Neurobio/MICROSCOPE/Shun/Project clamping/Recordings/202604-BiPOLES'

session_name = '20260520-SL432-stepTest_g0'
data_file_path = os.path.join(root_dir, f'{session_name}', f'data_{session_name}.mat')
timeseries_file_path = os.path.join(root_dir, f'{session_name}', f'timeseries_{session_name}.mat')
sync_file_path = os.path.join(root_dir, f'{session_name}', f'sync_{session_name}.mat')
save_path = os.path.join(root_dir, f'{session_name}', 'results')
if not os.path.exists(save_path):
    os.makedirs(save_path)
animal_name = session_name.split('-')[1]

# Load the MATLAB file
data_mat = mat73.loadmat(data_file_path)
timeseries_mat = mat73.loadmat(timeseries_file_path)
sync_mat = mat73.loadmat(sync_file_path)

# Extract signals


print(f"Data loaded: {len(dopamine_voltage)} samples")

ERROR:root:ERROR: MATLAB type not supported: string, (uint32)


Data loaded: 25061551 samples


### Analysis settings

In [ ]:
from collections import defaultdict
from pathlib import Path
import importlib.util
import sys
import types

try:
    import pyNeuroDAP as ndap
    if not hasattr(ndap, "get_traces") or not hasattr(ndap, "plot_sem"):
        raise ImportError("pyNeuroDAP imported without get_traces/plot_sem")
except (ModuleNotFoundError, ImportError) as exc:
    # Some local installs miss optional modeling deps imported by pyNeuroDAP.__init__.
    # Load the exact local modules needed here without importing the full package.
    repo_root = Path.cwd()
    package_dir = repo_root / "pyNeuroDAP"
    sys.modules.pop("pyNeuroDAP", None)
    pkg = types.ModuleType("pyNeuroDAP")
    pkg.__path__ = [str(package_dir)]
    sys.modules["pyNeuroDAP"] = pkg

    def _load_local_ndap_module(module_name):
        spec = importlib.util.spec_from_file_location(
            f"pyNeuroDAP.{module_name}",
            package_dir / f"{module_name}.py",
        )
        module = importlib.util.module_from_spec(spec)
        sys.modules[f"pyNeuroDAP.{module_name}"] = module
        spec.loader.exec_module(module)
        return module

    spikes_mod = _load_local_ndap_module("spikes")
    plots_mod = _load_local_ndap_module("plots")
    pkg.get_traces = spikes_mod.get_traces
    pkg.plot_sem = plots_mod.plot_sem
    ndap = pkg
    print(f"Loaded pyNeuroDAP get_traces/plot_sem directly because top-level import failed: {exc}")

# BrainClamp/brainclamp_gui.py LabJack mirror constants:
# volts = target_abs * (5.0 / 1023.0). The inverse gives the absolute
# BrainClamp target, then target_dff = target_abs / clamp_baseline_abs - 1.
LJ_ARDUINO_VREF = 5.0
LJ_ARDUINO_ADC_FULLSCALE = 1023.0
LJ_VOLTS_OFFSET = 0.0
LJ_VOLTS_PER_TARGET = LJ_ARDUINO_VREF / LJ_ARDUINO_ADC_FULLSCALE

# Protocol values from send_event_stepTest.py and send_event_rampTest.py.
STEP_AMPS = np.array([-0.20, -0.10, 0.10, 0.20, 0.50, 0.80])
STEP_HOLD_S = 5.0
STEP_BASELINE_S = 5.0
STEP_REPEATS_EACH = 30

RAMP_AMPS = np.array([0.20, -0.20, 0.10, -0.10])
RAMP_DUR_S = 2.0
RAMP_HOLD_S = 2.0
RAMP_PRE_WAIT_S = 0.4
RAMP_ITI_S = 5.0
RAMP_REPEATS_EACH = 30

SESSION_TYPE = "auto"  # "auto", "step", or "ramp"
ANALYSIS_FS = 200.0     # common analysis grid; matches BrainClamp PID/data cadence
CLAMP_ON_VOLTAGE_THRESHOLD = 0.01
TARGET_DFF_THRESHOLD = 0.035
AMP_MATCH_TOL = 0.04

STEP_ALIGN_WINDOW_S = (-1.0, 7.0)
RAMP_ALIGN_WINDOW_S = (-1.0, 6.5)
DA_BASELINE_WINDOW_S = (-0.5, -0.05)


### Convert clampTarget voltage and DA trace

In [ ]:
def _as_vector(x):
    """Flatten a MATLAB/mat73 channel to a 1D float vector."""
    if isinstance(x, dict) and "data" in x:
        x = x["data"]
    arr = np.asarray(x).squeeze()
    if arr.ndim != 1:
        arr = arr.reshape(-1)
    return arr.astype(float)


def _get_channel(data, names, existing_var=None):
    if existing_var is not None:
        return _as_vector(existing_var)
    for name in names:
        if name in data:
            return _as_vector(data[name])
    available = ", ".join(sorted(map(str, data.keys())))
    raise KeyError(f"Could not find any of {names}. Available data_mat keys: {available}")


clamp_target_voltage = _get_channel(
    data_mat,
    ["clampTarget", "clamp_target", "clampTargetVoltage", "clamp_target_voltage", "target"],
    existing_var=globals().get("clampTarget", None),
)

# Build a shared time base. This keeps the load cell untouched while allowing
# clampTarget to be present either at raw NI Fs or at an already downsampled rate.
recording_duration_s = len(dopamine_voltage) / float(Fs)
dopamine_fs = float(Fs)
clamp_target_fs = len(clamp_target_voltage) / recording_duration_s
common_fs = min(float(ANALYSIS_FS), dopamine_fs, clamp_target_fs)
common_t = np.arange(0, recording_duration_s, 1.0 / common_fs)

source_t_dopamine = np.arange(len(dopamine_voltage)) / dopamine_fs
source_t_target = np.arange(len(clamp_target_voltage)) / clamp_target_fs

dopamine_voltage_ds = np.interp(common_t, source_t_dopamine, dopamine_voltage)
clamp_target_voltage_ds = np.interp(common_t, source_t_target, clamp_target_voltage)

def rolling_median_dff(trace, fs, window_s=60.0):
    """Convert raw photometry F to fractional dF/F with a slow rolling median baseline."""
    x = np.asarray(trace, dtype=float)
    win = max(3, int(round(window_s * fs)))
    if win % 2 == 0:
        win += 1
    min_periods = max(3, min(win, int(round(5 * fs))))
    baseline = (
        pd.Series(x)
        .rolling(win, center=True, min_periods=min_periods)
        .median()
        .bfill()
        .ffill()
        .to_numpy()
    )
    eps = np.nanpercentile(np.abs(baseline), 1) * 1e-6
    eps = eps if np.isfinite(eps) and eps > 0 else 1e-9
    return (x - baseline) / np.maximum(np.abs(baseline), eps)


dopamine_dff = rolling_median_dff(dopamine_voltage_ds, common_fs)

target_abs = np.clip(
    (clamp_target_voltage_ds - LJ_VOLTS_OFFSET) / LJ_VOLTS_PER_TARGET,
    0,
    None,
)
clamp_on_mask = clamp_target_voltage_ds > CLAMP_ON_VOLTAGE_THRESHOLD

print(f"Session: {session_name}")
print(f"Common analysis Fs: {common_fs:.3f} Hz")
print(f"ClampTarget source Fs estimate: {clamp_target_fs:.3f} Hz")
print(f"ClampTarget voltage range: {np.nanmin(clamp_target_voltage_ds):.4f} to {np.nanmax(clamp_target_voltage_ds):.4f} V")


### Convert clampTarget to dF/F

In [ ]:
def contiguous_regions(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0 or not np.any(mask):
        return np.empty((0, 2), dtype=int)
    edges = np.diff(mask.astype(int))
    starts = list(np.where(edges == 1)[0] + 1)
    ends = list(np.where(edges == -1)[0] + 1)
    if mask[0]:
        starts = [0] + starts
    if mask[-1]:
        ends = ends + [mask.size]
    return np.asarray(list(zip(starts, ends)), dtype=int)


def merge_close_segments(segments, max_gap_samples):
    segments = np.asarray(segments, dtype=int)
    if segments.size == 0:
        return segments.reshape(0, 2)
    merged = [segments[0].tolist()]
    for start, end in segments[1:]:
        if start - merged[-1][1] <= max_gap_samples:
            merged[-1][1] = end
        else:
            merged.append([start, end])
    return np.asarray(merged, dtype=int)


def densest_level(x, bins=250):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    if x.size == 0:
        return np.nan
    lo, hi = np.nanpercentile(x, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return float(np.nanmedian(x))
    counts, edges = np.histogram(x, bins=bins, range=(lo, hi))
    if not np.any(counts):
        return float(np.nanmedian(x))
    i = int(np.argmax(counts))
    in_bin = (x >= edges[i]) & (x <= edges[i + 1])
    return float(np.nanmedian(x[in_bin])) if np.any(in_bin) else float((edges[i] + edges[i + 1]) / 2)


def estimate_episode_baseline_abs(target_abs, start, end, fs):
    duration_s = (end - start) / fs
    x = target_abs[start:end]
    if duration_s > 30:
        return densest_level(x)

    n_pre = max(1, int(round(0.25 * fs)))
    first = x[:n_pre]
    first = first[np.isfinite(first) & (first > 0)]
    if first.size >= max(3, n_pre // 4):
        return float(np.nanmedian(first))
    return densest_level(x)


def build_clamp_target_dff(target_abs, on_mask, fs):
    episodes = merge_close_segments(
        contiguous_regions(on_mask),
        max_gap_samples=int(round(0.25 * fs)),
    )
    min_episode_samples = int(round(0.5 * fs))
    target_dff = np.zeros_like(target_abs, dtype=float)
    rows = []

    for episode_i, (start, end) in enumerate(episodes):
        if end - start < min_episode_samples:
            continue
        baseline_abs = estimate_episode_baseline_abs(target_abs, start, end, fs)
        if not np.isfinite(baseline_abs) or baseline_abs <= 0:
            continue
        target_dff[start:end] = target_abs[start:end] / baseline_abs - 1.0
        rows.append({
            "session_name": session_name,
            "episode": episode_i,
            "start_idx": int(start),
            "end_idx": int(end),
            "start_s": start / fs,
            "end_s": end / fs,
            "duration_s": (end - start) / fs,
            "baseline_abs_target": baseline_abs,
        })

    return target_dff, pd.DataFrame(rows)


clampTarget_dff, clamp_episodes = build_clamp_target_dff(target_abs, clamp_on_mask, common_fs)

print(f"Clamp episodes found: {len(clamp_episodes)}")
display(clamp_episodes.head())


### Extract step/ramp events

In [ ]:
def nearest_protocol_amp(value, expected_amps, tol=AMP_MATCH_TOL):
    expected_amps = np.asarray(expected_amps, dtype=float)
    value = float(value)
    if expected_amps.size == 0 or not np.isfinite(value):
        return np.nan
    nearest = float(expected_amps[np.argmin(np.abs(expected_amps - value))])
    return nearest if abs(nearest - value) <= tol else float(np.round(value, 3))


def find_step_events(target_dff, fs, expected_amps=STEP_AMPS, threshold=TARGET_DFF_THRESHOLD):
    active = np.abs(target_dff) > threshold
    segments = merge_close_segments(
        contiguous_regions(active),
        max_gap_samples=int(round(0.15 * fs)),
    )
    min_samples = int(round(1.0 * fs))
    trim = int(round(0.2 * fs))
    rows = []
    for i, (start, end) in enumerate(segments):
        if end - start < min_samples:
            continue
        lo = min(end, start + trim)
        hi = max(lo + 1, end - trim)
        inner = target_dff[lo:hi]
        if inner.size == 0:
            inner = target_dff[start:end]
        amp_raw = float(np.nanmedian(inner))
        amp = nearest_protocol_amp(amp_raw, expected_amps)
        rows.append({
            "session_name": session_name,
            "session_type": "step",
            "kind": "step",
            "trial": len(rows) + 1,
            "amp": amp,
            "amp_raw": amp_raw,
            "start_idx": int(start),
            "end_idx": int(end),
            "start_s": start / fs,
            "end_s": end / fs,
            "duration_s": (end - start) / fs,
        })
    return pd.DataFrame(rows)


def find_ramp_events(
    target_dff,
    fs,
    expected_amps=RAMP_AMPS,
    slope_threshold_per_s=0.015,
    threshold=TARGET_DFF_THRESHOLD,
):
    y = np.asarray(target_dff, dtype=float)
    smooth_win = max(3, int(round(0.05 * fs)))
    y_smooth = (
        pd.Series(y)
        .rolling(smooth_win, center=True, min_periods=1)
        .median()
        .to_numpy()
    )
    slope = np.gradient(y_smooth) * fs
    slope_mask = np.abs(slope) > slope_threshold_per_s
    segments = merge_close_segments(
        contiguous_regions(slope_mask),
        max_gap_samples=int(round(0.30 * fs)),
    )

    min_samples = int(round(0.5 * fs))
    max_samples = int(round(4.0 * fs))
    level_win = max(1, int(round(0.25 * fs)))
    rows = []

    for start, end in segments:
        if end - start < min_samples or end - start > max_samples:
            continue
        pre = y[max(0, start - level_win):start]
        post = y[end:min(y.size, end + level_win)]
        start_level = float(np.nanmedian(pre)) if pre.size else float(y_smooth[start])
        end_level = float(np.nanmedian(post)) if post.size else float(y_smooth[min(end, y.size - 1)])
        delta = end_level - start_level
        if abs(delta) < threshold:
            continue

        if abs(start_level) <= abs(end_level):
            phase = "away_from_baseline"
            amp_raw = end_level
        else:
            phase = "return_to_baseline"
            amp_raw = start_level

        amp = nearest_protocol_amp(amp_raw, expected_amps)
        rows.append({
            "session_name": session_name,
            "session_type": "ramp",
            "kind": "ramp",
            "ramp_phase": phase,
            "trial": len(rows) // 2 + 1,
            "amp": amp,
            "amp_raw": amp_raw,
            "start_level": start_level,
            "end_level": end_level,
            "delta": delta,
            "start_idx": int(start),
            "end_idx": int(end),
            "start_s": start / fs,
            "end_s": end / fs,
            "duration_s": (end - start) / fs,
        })
    return pd.DataFrame(rows)


step_candidates = find_step_events(clampTarget_dff, common_fs)
ramp_candidates = find_ramp_events(clampTarget_dff, common_fs)

def infer_session_type(requested, session_name, step_events, ramp_events):
    requested = str(requested).lower()
    if requested in {"step", "ramp"}:
        return requested
    name = str(session_name).lower()
    if "ramp" in name:
        return "ramp"
    if "step" in name:
        return "step"
    if len(ramp_events) >= max(2, len(step_events)):
        return "ramp"
    return "step"


session_type = infer_session_type(SESSION_TYPE, session_name, step_candidates, ramp_candidates)
step_ramp_events = (ramp_candidates if session_type == "ramp" else step_candidates).copy()

print(f"Inferred session type: {session_type}")
print(f"Step candidates: {len(step_candidates)} | Ramp candidates: {len(ramp_candidates)}")
if len(step_ramp_events):
    display(step_ramp_events.groupby(["kind", "amp"]).size().rename("n").reset_index())
    display(step_ramp_events.head(12))
else:
    print("No step/ramp events found. Check CLAMP_ON_VOLTAGE_THRESHOLD and TARGET_DFF_THRESHOLD.")


### Preview extracted events

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.2))
stride = max(1, int(round(common_fs / 10)))
ax.plot(common_t[::stride] / 60, clampTarget_dff[::stride], color="0.25", linewidth=1)

if len(step_ramp_events):
    cmap = plt.get_cmap("coolwarm")
    amps = np.sort(step_ramp_events["amp"].dropna().unique())
    max_abs_amp = max(0.01, np.nanmax(np.abs(amps))) if len(amps) else 1.0
    for amp in amps:
        ev = step_ramp_events[np.isclose(step_ramp_events["amp"], amp)]
        color = cmap((amp / max_abs_amp + 1) / 2)
        y = np.full(len(ev), amp)
        ax.scatter(ev["start_s"] / 60, y, s=16, color=color, label=f"{amp:+.2f}")

ax.axhline(0, color="0.7", linewidth=0.8)
ax.set_title(f"{session_name} | {session_type} session | clampTarget_dff events")
ax.set_xlabel("Time (min)")
ax.set_ylabel("clampTarget dF/F0")
if len(step_ramp_events):
    ax.legend(title="amp", ncol=min(6, len(amps)), fontsize=8, frameon=False)
fig.tight_layout()


### Plot aligned DA traces

In [ ]:
def baseline_subtract_traces(traces, t_axis, baseline_window=DA_BASELINE_WINDOW_S):
    traces = np.asarray(traces, dtype=float)
    t_axis = np.asarray(t_axis, dtype=float)
    baseline_mask = (t_axis >= baseline_window[0]) & (t_axis <= baseline_window[1])
    if not np.any(baseline_mask):
        return traces
    baseline = np.nanmean(traces[:, baseline_mask], axis=1, keepdims=True)
    return traces - baseline


def amp_colors(amps):
    amps = np.asarray(list(amps), dtype=float)
    max_abs = max(0.01, float(np.nanmax(np.abs(amps)))) if amps.size else 1.0
    cmap = plt.get_cmap("coolwarm")
    return {float(amp): cmap((float(amp) / max_abs + 1) / 2) for amp in amps}


def plot_aligned_da_by_amp(events, trace, target_trace, fs, time_window, title, phase=None, ax=None):
    data = events.copy()
    if phase is not None and "ramp_phase" in data.columns:
        data = data[data["ramp_phase"] == phase]
    if data.empty:
        if ax is None:
            _, ax = plt.subplots(figsize=(6, 4))
        ax.set_title(f"{title}: no events")
        return ax

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))

    amps = np.sort(data["amp"].dropna().unique())
    colors = amp_colors(amps)
    ax_target = ax.twinx()

    for amp in amps:
        ev = data[np.isclose(data["amp"], amp)]
        event_idx = ev["start_idx"].to_numpy(dtype=int)
        da_traces, t_axis = ndap.get_traces(
            trace,
            event_idx,
            time_range=time_window,
            signal_fs=fs,
            event_system="ni",
            signal_system="ni",
            fill_value=np.nan,
        )
        da_traces = baseline_subtract_traces(da_traces, t_axis)
        color = colors[float(amp)]
        ndap.plot_sem(
            da_traces,
            x=t_axis,
            ax=ax,
            color=color,
            label=f"{amp:+.2f} (n={len(ev)})",
            fill=True,
            plot_individual=False,
        )

        target_traces, _ = ndap.get_traces(
            target_trace,
            event_idx,
            time_range=time_window,
            signal_fs=fs,
            event_system="ni",
            signal_system="ni",
            fill_value=np.nan,
        )
        ax_target.plot(
            t_axis,
            np.nanmean(target_traces, axis=0),
            color=color,
            alpha=0.55,
            linewidth=1,
            linestyle="--",
            label="_nolegend_",
        )

    ax.axvline(0, color="0.2", linewidth=0.8, linestyle=":")
    ax.axhline(0, color="0.75", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Time from target event (s)")
    ax.set_ylabel("DA dF/F0, baseline-subtracted")
    ax_target.set_ylabel("clampTarget dF/F0")
    ax.legend(frameon=False, fontsize=8)
    return ax


In [ ]:
if step_ramp_events.empty:
    print("No events to plot.")
elif session_type == "step":
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_aligned_da_by_amp(
        step_ramp_events,
        dopamine_dff,
        clampTarget_dff,
        common_fs,
        STEP_ALIGN_WINDOW_S,
        title=f"{session_name} | step onset aligned DA",
        ax=ax,
    )
    fig.tight_layout()
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    plot_aligned_da_by_amp(
        step_ramp_events,
        dopamine_dff,
        clampTarget_dff,
        common_fs,
        RAMP_ALIGN_WINDOW_S,
        title=f"{session_name} | ramp away from baseline",
        phase="away_from_baseline",
        ax=axes[0],
    )
    plot_aligned_da_by_amp(
        step_ramp_events,
        dopamine_dff,
        clampTarget_dff,
        common_fs,
        RAMP_ALIGN_WINDOW_S,
        title=f"{session_name} | ramp return to baseline",
        phase="return_to_baseline",
        ax=axes[1],
    )
    fig.tight_layout()


### Optional export

In [ ]:
# Optional: save the extracted event table for downstream analysis.
# output_csv = os.path.join(save_path, f"{session_name}_{session_type}_step_ramp_events.csv")
# step_ramp_events.to_csv(output_csv, index=False)
# print(f"Saved events to {output_csv}")
